In [ ]:
pip install -q datasets langchain-nvidia-ai-endpoints langchain-core langchain-community langchain-experimental pypdf

In [ ]:
from datasets import load_dataset
dataset = load_dataset("PatronusAI/financebench")
dataset

## 📄 NVIDIA Document Ingestion, Semantic Chunking & .abatch() Metadata Enrichment Pipeline (40 RPM Throttled)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import importlib
from dotenv import load_dotenv
import document_pipeline
importlib.reload(document_pipeline)
from document_pipeline import DocumentPipeline

load_dotenv()

# Initialize Pipeline with NVIDIA Builder models & strict 40 RPM Rate Limiter
pipeline = DocumentPipeline(
    llm_model="meta/llama-3.1-8b-instruct",
    embedding_model="nvidia/nv-embedqa-e5-v5",
    temperature=0.1,
    breakpoint_threshold_type="percentile",
    batch_size=5,
    max_rpm=40
)

# Execute Complete Pipeline (Loading -> Doc Meta -> Semantic Chunking -> .abatch() Enrichment [40 RPM] -> Storage)
enriched_chunks = await pipeline.run_pipeline_async(data_dir="Data")

In [ ]:
import json
import pandas as pd

with open("Data/processed_documents.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame([
    {
        "ID": item["id"],
        "Title": item["metadata"].get("chunk_title"),
        "Summary": item["metadata"].get("chunk_summary"),
        "Entities": ", ".join(item["metadata"].get("entities", [])),
        "Mandates": len(item["metadata"].get("compliance_mandates", [])),
        "Page": item["metadata"].get("page")
    }
    for item in data
])
df.head(10)

In [ ]:
# Query the Enriched Vector Store
query = "What are the rules regarding loan disbursals and fees paid to LSPs?"
results = pipeline.search(query, k=3)

for i, res in enumerate(results, 1):
    print(f"\n--- Result {i}: {res.metadata.get('chunk_title')} ---")
    print(f"Summary: {res.metadata.get('chunk_summary')}")
    print(f"Compliance Mandates: {res.metadata.get('compliance_mandates')}")
    print(f"Entities: {res.metadata.get('entities')}")
    print(f"Content: {res.page_content[:200]}...")